In [1]:
import os
import pandas as pd

# Paths
base_dir = r'../dataset/Dataset'
image_dir = os.path.join(base_dir, 'Memes')  # Path to the extracted "Memes" folder
input_excel = os.path.join(base_dir, 'multi-sent.xlsx')  # Path to the Excel file
output_csv = "../working/image_caption_labels.csv"  # Path for the output CSV file

# Step 1: Get all image files from the Memes folder   
image_files = os.listdir(image_dir)

# Step 2: Load the Excel file
df = pd.read_excel(input_excel)

# Step 3: Create a new list to store data for CSV
image_data = []

# Step 4: Iterate through the Excel data and match image_name with files
for index, row in df.iterrows():
    image_name = row['image_name']  # Assuming the column name is 'image_name'

    # Check if the image exists in the folder
    if image_name in image_files:
        # Construct the full image path
        image_path = os.path.join(image_dir, image_name)

        # Extract Captions and Label_Sentiment (adjust column names if necessary)
        caption = row['Captions']  # Assuming 'Captions' is the column name for captions
        label_sentiment = row['Label_Sentiment']  # Assuming 'Label_Sentiment' is the column name

        # Append the row with image path, image name, captions, and sentiment label
        image_data.append([image_path, image_name, caption, label_sentiment])

# Step 5: Create a DataFrame for the matched data
image_df = pd.DataFrame(image_data, columns=['Image_path', 'image_name', 'Captions', 'Label_Sentiment'])

# Step 6: Save the DataFrame to a CSV file
image_df.to_csv(output_csv, index=False)

print(f"CSV file saved at {output_csv}")


CSV file saved at ../working/image_caption_labels.csv


In [2]:
import pandas as pd

# Path to the CSV file (replace this with the actual path if different)
csv_file = "../working/image_caption_labels.csv"

# Step 1: Load the CSV file into a DataFrame
df = pd.read_csv(csv_file)

# Step 2: Print the DataFrame to show the contents
df.head()


,Image_path,image_name,Captions,Label_Sentiment
0,../dataset/Dataset\Memes\tangaila (1).jpg,tangaila (1).jpg,"আম্মাঃ HSC চলে আসছে , এখন থেকে তোর মোবাইল , ল্...",positive
1,../dataset/Dataset\Memes\tangaila (2).jpg,tangaila (2).jpg,WHEN YOUR COUSINS TAKES YOU TO THE DHAN KHET A...,negative
2,../dataset/Dataset\Memes\tangaila (3).jpg,tangaila (3).jpg,WHEN HE SAID 10 MINUTES BUT IT WAS ONLY 2 MINUTES,negative
3,../dataset/Dataset\Memes\tangaila (4).jpg,tangaila (4).jpg,SHE - I CAN'T BE WITH YOU -তবে শেষবারের মত দ...,negative
4,../dataset/Dataset\Memes\tangaila (5).jpg,tangaila (5).jpg,"যখন কোন Teacher বলে ""সত্যটা বলো, তাহলে আর কি...",negative


In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Step 2: Split the data into train (70%), test (20%), and validation (10%) using stratified splitting
train_df, temp_df = train_test_split(df, test_size=0.3, stratify=df['Label_Sentiment'], random_state=42)
test_df, val_df = train_test_split(temp_df, test_size=1/3, stratify=temp_df['Label_Sentiment'], random_state=42)

# Step 3: Save the split data into separate CSV files
train_df.to_csv('../working/train.csv', index=False)
test_df.to_csv('../working/test.csv', index=False)
val_df.to_csv('../working/validation.csv', index=False)

# Print the shapes of the resulting datasets for verification
print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")
print(f"Validation shape: {val_df.shape}")


Train shape: (3057, 4)
Test shape: (874, 4)
Validation shape: (437, 4)


In [4]:
import pandas as pd
import re
import string

# Function to remove punctuation (preserve Bangla characters)
def remove_punctuation(text):
    if pd.isna(text):
        return text
    return text.translate(str.maketrans('', '', string.punctuation))

# Function to remove extra whitespace
def remove_whitespace(text):
    if pd.isna(text):
        return text
    return " ".join(text.split())

# Function to remove emojis
def remove_emojis(text):
    if pd.isna(text):
        return text
    emoji_pattern = re.compile(
        "["
        u"\U0001F600-\U0001F64F"  # emoticons
        u"\U0001F300-\U0001F5FF"  # symbols & pictographs
        u"\U0001F680-\U0001F6FF"  # transport & map symbols
        u"\U0001F1E0-\U0001F1FF"  # flags (iOS)
        u"\U00002702-\U000027B0"
        u"\U000024C2-\U0001F251"
        "]+", flags=re.UNICODE)
    return emoji_pattern.sub(r'', text)

# Function to remove URLs
def remove_urls(text):
    if pd.isna(text):
        return text
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

# Function to remove HTML tags
def remove_html(text):
    if pd.isna(text):
        return text
    html_pattern = re.compile(r'<.*?>')
    return html_pattern.sub(r'', text)

# Function to remove special characters (preserve Bangla characters)
def remove_special_characters(text):
    if pd.isna(text):
        return text
    return re.sub(r'[^A-Za-z0-9\s\u0980-\u09FF]', '', text)

# Combine all cleaning functions
def clean_text(text):
    text = remove_urls(text)
    text = remove_html(text)
    text = remove_emojis(text)
    text = remove_punctuation(text)
    text = remove_special_characters(text)
    text = remove_whitespace(text)
    return text

# Mapping categories to integers
category_mapping = {
    'positive': 0,
    'negative': 1,
    'neutral': 2,
}

# Paths to the previously saved CSVs
csv_paths = {
    'Train': '../working/train.csv',
    'Test': '../working/test.csv',
    'Validation': '../working/validation.csv'
}

# Output paths for the cleaned CSVs
cleaned_output_paths = {
    'Train': '../working/train_cleaned.csv',
    'Test': '../working/test_cleaned.csv',
    'Validation': '../working/validation_cleaned.csv'
}

# Text columns to clean
text_columns = ['Captions', 'Label_Sentiment']

# Loop through each dataset
for key in csv_paths:
    # Load the dataset
    df = pd.read_csv(csv_paths[key])

    # Apply cleaning to all relevant text columns
    for column in text_columns:
        df[column] = df[column].astype(str).apply(clean_text)

    # Map the 'Label_Sentiment' column to integers
    df['Label_Sentiment'] = df['Label_Sentiment'].map(category_mapping)

    # Add a 'label' column (same as 'Label_Sentiment' for now)
    df['label'] = df['Label_Sentiment']

    # Display the cleaned dataframe
    print(f"Cleaned {key} dataframe:")
    print(df.head())

    # Save the cleaned dataframe to a new CSV file
    df.to_csv(cleaned_output_paths[key], index=False)
    print(f"Cleaned dataframe saved to {cleaned_output_paths[key]}\n")


Cleaned Train dataframe:
                                        Image_path              image_name  \
0  ../dataset/Dataset\Memes\bamboo-vaiya (115).jpg  bamboo-vaiya (115).jpg   
1            ../dataset/Dataset\Memes\KAM (61).png            KAM (61).png   
2           ../dataset/Dataset\Memes\KAM (345).jpg           KAM (345).jpg   
3           ../dataset/Dataset\Memes\KAM (572).jpg           KAM (572).jpg   
4           ../dataset/Dataset\Memes\KAM (714).jpg           KAM (714).jpg   

                                            Captions  Label_Sentiment  label  
0  WHEN GIRL CHANGES HER RELATIONSHIP STATUS TO S...                1      1  
1  ভাল ফ্যাকাল্টি পাইলে আমিও ভাল গ্রেড উঠাইতাম কি...                0      0  
2  BREAK UP এরপর মেয়েরা যে গান গায় এনালগ যুগঃ যেও...                1      1  
3  কি হয়সে ভাই আপনার মন খারাপ কেন সেই কখন থেকে মো...                0      0  
4  পড়তে বসার সময় বসার 10 মিনিট পর বসার 20 মিনিট...                0      0  
Cleaned dataframe saved to ../wo

In [5]:
train_df = pd.read_csv('../working/train_cleaned.csv')
train_df.head()

,Image_path,image_name,Captions,Label_Sentiment,label
0,../dataset/Dataset\Memes\bamboo-vaiya (115).jpg,bamboo-vaiya (115).jpg,WHEN GIRL CHANGES HER RELATIONSHIP STATUS TO S...,1,1
1,../dataset/Dataset\Memes\KAM (61).png,KAM (61).png,ভাল ফ্যাকাল্টি পাইলে আমিও ভাল গ্রেড উঠাইতাম কি...,0,0
2,../dataset/Dataset\Memes\KAM (345).jpg,KAM (345).jpg,BREAK UP এরপর মেয়েরা যে গান গায় এনালগ যুগঃ যেও...,1,1
3,../dataset/Dataset\Memes\KAM (572).jpg,KAM (572).jpg,কি হয়সে ভাই আপনার মন খারাপ কেন সেই কখন থেকে মো...,0,0
4,../dataset/Dataset\Memes\KAM (714).jpg,KAM (714).jpg,পড়তে বসার সময় বসার 10 মিনিট পর বসার 20 মিনিট...,0,0


In [6]:
test_df = pd.read_csv('../working/test_cleaned.csv')
test_df.head()

,Image_path,image_name,Captions,Label_Sentiment,label
0,../dataset/Dataset\Memes\chosha (31).jpg,chosha (31).jpg,Hi সরি আমার বয়ফ্রেন্ড আছে আমি Hi বলছি বাঁড়া চু...,1,1
1,../dataset/Dataset\Memes\nurani-memes (219).jpg,nurani-memes (219).jpg,Actual introverts Extroverts pretending to be ...,1,1
2,../dataset/Dataset\Memes\nurani-memes (47).jpg,nurani-memes (47).jpg,Types of Headache Migraine Hypertension Stress...,1,1
3,../dataset/Dataset\Memes\bamboo-vaiya (18).jpg,bamboo-vaiya (18).jpg,এই আপুকে একটা খারাপ জীন আছর করছে সবাই তার জন্য...,1,1
4,../dataset/Dataset\Memes\ovodro_img (539).jpg,ovodro_img (539).jpg,আমার এবার তিনজন ছেলেকে বিয়ে করা হবে এরম বারে ব...,1,1


In [7]:
validation_df = pd.read_csv('../working/validation_cleaned.csv')
validation_df.head()

,Image_path,image_name,Captions,Label_Sentiment,label
0,../dataset/Dataset\Memes\tangaila (242).jpg,tangaila (242).jpg,If গাঞ্জাখোর had a face,1,1
1,../dataset/Dataset\Memes\bamboo-vaiya (447).jpg,bamboo-vaiya (447).jpg,যখন কোনো মাদারচোদ ঘুড়ি উরায়,1,1
2,../dataset/Dataset\Memes\tangaila (172).jpg,tangaila (172).jpg,বিন্দুবাসিনী বয়েজের সসচ এর রেজাল্ট বিন্দুবাসিন...,1,1
3,../dataset/Dataset\Memes\bamboo-vaiya (52).jpg,bamboo-vaiya (52).jpg,When You Musterbate for Several Time জ্বলে রে ...,1,1
4,../dataset/Dataset\Memes\KAM (408).jpg,KAM (408).jpg,বাংলাদেশআমাদের রুবেল আছে ইন্ডিয়া আমাদের ICC আছে,0,0


In [8]:
#!pip install torch torchvision transformers pillow tqdm pandas numpy scikit-learn matplotlib openpyxl ipykernel

In [9]:
#!pip install transformers

In [10]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from transformers import BertTokenizer, BertModel
from torch.optim import AdamW
import torchvision.models as models
from tqdm import tqdm

In [11]:
from transformers import AutoImageProcessor, SwiftFormerModel
import torch
image_processor = AutoImageProcessor.from_pretrained("MBZUAI/swiftformer-xs")

model = SwiftFormerModel.from_pretrained("MBZUAI/swiftformer-xs")

Loading weights:   0%|          | 0/307 [00:00<?, ?it/s]

[transformers] SwiftFormerModel LOAD REPORT from: MBZUAI/swiftformer-xs
Key                      | Status     |  | 
-------------------------+------------+--+-
norm.weight              | UNEXPECTED |  | 
head.weight              | UNEXPECTED |  | 
norm.running_mean        | UNEXPECTED |  | 
head.bias                | UNEXPECTED |  | 
dist_head.bias           | UNEXPECTED |  | 
norm.num_batches_tracked | UNEXPECTED |  | 
dist_head.weight         | UNEXPECTED |  | 
norm.bias                | UNEXPECTED |  | 
norm.running_var         | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [12]:
# Initialize BERT tokenizer and model
bert_tokenizer = BertTokenizer.from_pretrained('bert-base-multilingual-cased')
bert_model = BertModel.from_pretrained("bert-base-multilingual-cased")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [13]:
# Check if GPU isA available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

Using device: cuda


In [16]:
import os

# Enable device-side assertions
os.environ['TORCH_USE_CUDA_DSA'] = '1'

In [17]:
model.to(device)

SwiftFormerModel(
  (patch_embed): SwiftFormerPatchEmbedding(
    (patch_embedding): Sequential(
      (0): Conv2d(3, 24, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (1): BatchNorm2d(24, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU()
      (3): Conv2d(24, 48, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (4): BatchNorm2d(48, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (5): ReLU()
    )
  )
  (encoder): SwiftFormerEncoder(
    (network): ModuleList(
      (0): SwiftFormerStage(
        (blocks): ModuleList(
          (0-1): 2 x SwiftFormerConvEncoder(
            (depth_wise_conv): Conv2d(48, 48, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=48)
            (norm): BatchNorm2d(48, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (point_wise_conv1): Conv2d(48, 192, kernel_size=(1, 1), stride=(1, 1))
            (act): GELU(approximate='none')
            (point_wise

In [18]:
bert_model.to(device)

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(119547, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
 

In [19]:
from torchvision import transforms
from PIL import Image
from torch.utils.data import Dataset

max_seq_length = 512  # Set your desired maximum sequence length for BERT

# Define the pre-processing transformations for images
transform = transforms.Compose([
    transforms.Resize(224),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class MyMultimodalDataset(Dataset):
    def __init__(self, data, transform=None, tokenizer=None, max_seq_length=512):
        self.data = data
        self.transform = transform
        self.tokenizer = tokenizer
        self.max_seq_length = max_seq_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        image_path = self.data.iloc[idx]['Image_path']
        try:
            image = Image.open(image_path).convert('RGB')
            if self.transform is not None:
                image = self.transform(image)
        except Exception as e:
            print(f"Error loading image at index {idx}: {e}")
            return None, None, None, None

        if image is None:
            return None, None, None, None

        context = self.data.iloc[idx]['Captions']

        inputs = self.tokenizer(context, padding='max_length', truncation=True, max_length=self.max_seq_length, return_tensors='pt')
        input_ids = inputs['input_ids']
        attention_mask = inputs['attention_mask']

        label = self.data.iloc[idx]['label']

        return image, input_ids, attention_mask, label
    

In [20]:
# Create custom datasets with MyMultimodalDataset
train_dataset = MyMultimodalDataset(train_df, transform=transform, tokenizer=bert_tokenizer, max_seq_length=max_seq_length)
test_dataset = MyMultimodalDataset(test_df, transform=transform, tokenizer=bert_tokenizer, max_seq_length=max_seq_length)
val_dataset = MyMultimodalDataset(validation_df, transform=transform, tokenizer=bert_tokenizer, max_seq_length=max_seq_length)

# Define data loaders
batch_size = 8  # Set your desired batch size
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [22]:
import torch
import torchvision.models as models
from transformers import AutoModel, AutoTokenizer
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from tqdm import tqdm
import time
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# Assuming you have defined your train_loader, val_loader, optimizer, criterion, model, bert_model, etc.

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Define the regressor model and optimizer
regressor = torch.nn.Sequential(
    torch.nn.Linear(11548, 512),  # Adjusted input dimension
    torch.nn.ReLU(),
    torch.nn.Dropout(0.5),
    torch.nn.Linear(512, 3)
).to(device)

optimizer = torch.optim.AdamW(regressor.parameters(), lr=0.001)
criterion = torch.nn.CrossEntropyLoss()

num_epochs = 35
train_losses = []
val_losses = []

# Training time
start_time = time.time()

for epoch in range(num_epochs):
    running_train_loss = 0.0
    
    regressor.train()

    for images, texts, attention_masks, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        input_ids = texts.squeeze(1).to(device)
        attention_mask = attention_masks.squeeze(1).to(device)

        optimizer.zero_grad()

        # Get image features
        with torch.no_grad():
            outputs_image = model(pixel_values=images)
        img_hidden_states = outputs_image.last_hidden_state
        # Flatten the image features
        img_feats = img_hidden_states.view(img_hidden_states.size(0), -1)

        # Get text features
        outputs_text = bert_model(input_ids=input_ids, attention_mask=attention_mask)
        text_hidden_states = outputs_text.last_hidden_state
        text_feats = text_hidden_states[:, 0, :]

        # Concatenate image and text features
        combined_feats = torch.cat((img_feats, text_feats), dim=1)

        # Forward pass
        predictions = regressor(combined_feats)
        loss = criterion(predictions, labels)

        # Backward pass
        loss.backward()
        optimizer.step()

        running_train_loss += loss.item()

    epoch_train_loss = running_train_loss / len(train_loader)
    train_losses.append(epoch_train_loss)

    print(f"Epoch [{epoch + 1}/{num_epochs}] - Train Loss: {epoch_train_loss:.4f}")

end_time = time.time()
execution_time = end_time - start_time
print(f"Total execution time: {execution_time:.2f} seconds")

Epoch [1/35] - Train Loss: 0.8184
Epoch [2/35] - Train Loss: 0.6167
Epoch [3/35] - Train Loss: 0.4044
Epoch [4/35] - Train Loss: 0.2659
Epoch [5/35] - Train Loss: 0.2095
Epoch [6/35] - Train Loss: 0.1783
Epoch [7/35] - Train Loss: 0.1261
Epoch [8/35] - Train Loss: 0.1061
Epoch [9/35] - Train Loss: 0.1072
Epoch [10/35] - Train Loss: 0.1395
Epoch [11/35] - Train Loss: 0.0996
Epoch [12/35] - Train Loss: 0.0880
Epoch [13/35] - Train Loss: 0.0981
Epoch [14/35] - Train Loss: 0.0944
Epoch [15/35] - Train Loss: 0.1125
Epoch [16/35] - Train Loss: 0.0795
Epoch [17/35] - Train Loss: 0.0527
Epoch [18/35] - Train Loss: 0.0678
Epoch [19/35] - Train Loss: 0.0525
Epoch [20/35] - Train Loss: 0.0771
Epoch [21/35] - Train Loss: 0.0901
Epoch [22/35] - Train Loss: 0.0794
Epoch [23/35] - Train Loss: 0.0820
Epoch [24/35] - Train Loss: 0.0600
Epoch [25/35] - Train Loss: 0.0668
Epoch [26/35] - Train Loss: 0.0743
Epoch [27/35] - Train Loss: 0.0398
Epoch [28/35] - Train Loss: 0.0675
Epoch [29/35] - Train Loss: 0

In [23]:
import torch
from tqdm import tqdm
import time

# Set models to evaluation mode
model.eval()
bert_model.eval()

# Prepare lists to store predicted and true labels
predicted_labels = []
true_labels = []

# Set your device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Test loop
start_time = time.time()

with torch.no_grad():
    for test_images, test_texts, test_attention_masks, test_labels in tqdm(test_loader, desc='Testing', leave=False):
        test_images = test_images.to(device)
        test_labels = (test_labels).to(device)  # Convert labels from 1-5 to 0-4
        test_texts = test_texts.to(device)
        test_attention_masks = test_attention_masks.to(device)

        optimizer.zero_grad()

        # Extract features from image-based model
        test_img_feats = model(test_images)

        # Extract features from text-based model (BERT)
        test_texts = test_texts.squeeze(1)
        test_attention_masks = test_attention_masks.squeeze(1)
        test_outputs = bert_model(input_ids=test_texts, attention_mask=test_attention_masks)
        
        # Extract relevant features for concatenation
        # Handle tensor manipulation for compatibility
        # Reshape test_img_feats if compatible
        test_img_feats = test_img_feats[0]  # Access a specific part of the BaseModelOutputWithNoAttention
        test_img_feats = test_img_feats.reshape(test_img_feats.shape[0], -1)  # Reshape if compatible

        # Extract representations from text-based model
        test_text_feats = test_outputs.last_hidden_state[:, 0, :]  # Select appropriate representations

        # Combine features early
        combined_feats = torch.cat((test_img_feats, test_text_feats), dim=1)

        # Classify combined features
        combined_classifier = torch.nn.Sequential(
            torch.nn.Linear(combined_feats.shape[1], 512).to(device),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.5),
            torch.nn.Linear(512, 3).to(device),  # Change the output size to 5 for 5 labels
        )

        combined_logits = combined_classifier(combined_feats)
        test_predictions = torch.nn.functional.softmax(combined_logits, dim=1)
        predicted_classes = torch.argmax(test_predictions, dim=1)  # Revert back to labels from 1-5, Add 1 here 

        predicted_labels.extend(predicted_classes.cpu().numpy())
        true_labels.extend(test_labels.cpu().numpy().tolist())

end_time = time.time()
execution_time = end_time - start_time

# Print or use the predicted labels and true labels as needed
print("Predicted Labels:", predicted_labels)
print("True Labels:", true_labels)
print(f"Total execution time for testing: {execution_time:.2f} seconds")

Predicted Labels: [np.int64(2), np.int64(1), np.int64(1), np.int64(2), np.int64(1), np.int64(1), np.int64(2), np.int64(1), np.int64(1), np.int64(2), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(2), np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(2), np.int64(2), np.int64(2), np.int64(0), np.int64(2), np.int64(0), np.int64(2), np.int64(2), np.int64(2), np.int64(1), np.int64(1), np.int64(0), np.int64(1), np.int64(1), np.int64(2), np.int64(1), np.int64(2), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(2), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(0), np.int64(2), np.int64(1), np.int64(2), np.int64(2), np.int64(0), np.int64(2), np.int64(1), np.int64(2), np.int64(2), np.int64(0), np.int64(2), np.int64(1), np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(2), np.int64(0), np.int64(0), np.int

In [24]:
predicted_labels

[np.int64(2),
 np.int64(1),
 np.int64(1),
 np.int64(2),
 np.int64(1),
 np.int64(1),
 np.int64(2),
 np.int64(1),
 np.int64(1),
 np.int64(2),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(2),
 np.int64(1),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(1),
 np.int64(0),
 np.int64(0),
 np.int64(2),
 np.int64(2),
 np.int64(2),
 np.int64(0),
 np.int64(2),
 np.int64(0),
 np.int64(2),
 np.int64(2),
 np.int64(2),
 np.int64(1),
 np.int64(1),
 np.int64(0),
 np.int64(1),
 np.int64(1),
 np.int64(2),
 np.int64(1),
 np.int64(2),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(1),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(2),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(0),
 np.int64(2),
 np.int64(1),
 np.int64(2),
 np.int64(2),
 np.int64(0),
 np.int64(2),
 np.int64(1),
 np.int64(2),
 np.int64(2),
 np.int64(0),
 np.int64(2),
 np.int64(1),
 np.int64(1),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.in

In [25]:
true_labels

[1,
 1,
 1,
 1,
 1,
 0,
 1,
 1,
 0,
 1,
 1,
 2,
 1,
 1,
 2,
 1,
 1,
 0,
 1,
 0,
 1,
 1,
 1,
 2,
 1,
 1,
 0,
 1,
 1,
 1,
 0,
 1,
 0,
 1,
 1,
 1,
 1,
 0,
 1,
 1,
 2,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 2,
 1,
 1,
 1,
 0,
 0,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 1,
 1,
 1,
 0,
 1,
 0,
 0,
 1,
 2,
 0,
 2,
 1,
 1,
 1,
 0,
 1,
 0,
 1,
 0,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 0,
 1,
 2,
 1,
 1,
 1,
 2,
 0,
 1,
 1,
 0,
 0,
 0,
 2,
 1,
 1,
 1,
 1,
 0,
 1,
 1,
 0,
 2,
 1,
 0,
 1,
 0,
 0,
 1,
 1,
 1,
 1,
 1,
 0,
 1,
 1,
 1,
 0,
 1,
 1,
 0,
 1,
 1,
 1,
 1,
 1,
 0,
 1,
 1,
 1,
 0,
 2,
 0,
 1,
 2,
 1,
 0,
 1,
 0,
 2,
 0,
 1,
 0,
 1,
 1,
 1,
 0,
 0,
 2,
 1,
 0,
 1,
 1,
 1,
 0,
 1,
 1,
 1,
 1,
 1,
 0,
 0,
 1,
 0,
 1,
 1,
 1,
 1,
 2,
 0,
 0,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 1,
 1,
 0,
 1,
 0,
 1,
 1,
 0,
 1,
 1,
 0,
 1,
 1,
 1,
 0,
 1,
 1,
 0,
 0,
 1,
 0,
 1,
 1,
 0,
 1,
 1,
 0,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 1,
 1,
 1,
 1,
 0,
 1,


In [26]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, mean_squared_error, classification_report

# Calculate accuracy
accuracy = accuracy_score(true_labels, predicted_labels)

# Calculate precision, recall, F1-score overall (macro average)
precision, recall, f1_score_macro, _ = precision_recall_fscore_support(true_labels, predicted_labels, average='macro')

# Calculate weighted precision, recall, and F1-score
precision_weighted, recall_weighted, f1_score_weighted, _ = precision_recall_fscore_support(true_labels, predicted_labels, average='weighted')

# Calculate confusion matrix
conf_matrix = confusion_matrix(true_labels, predicted_labels)

# Calculate Mean Squared Error
mse = mean_squared_error(true_labels, predicted_labels)

# Calculate Sensitivity (Recall) for each class
sensitivity_per_class = recall
# Calculate Specificity for each class
specificity_per_class = []
for i in range(len(conf_matrix)):
    tn = np.sum(conf_matrix) - (np.sum(conf_matrix[i, :]) + np.sum(conf_matrix[:, i]) - conf_matrix[i, i])
    fp = np.sum(conf_matrix[:, i]) - conf_matrix[i, i]
    specificity_per_class.append(tn / (tn + fp))

# Print overall calculated metrics
print(f"Accuracy: {accuracy}")
print(f"Precision (macro): {precision}")
print(f"Recall (macro): {recall}")
print(f"F1-Score (macro): {f1_score_macro}")
print(f"Weighted F1-Score: {f1_score_weighted}")
print(f"Mean Squared Error: {mse}")

# Print Sensitivity and Specificity for each class
print(f"Sensitivity (Recall) for each class: {sensitivity_per_class}")
print(f"Specificity for each class: {specificity_per_class}")


Accuracy: 0.38558352402745993
Precision (macro): 0.3614316301820884
Recall (macro): 0.351435495113656
F1-Score (macro): 0.32327924151863635
Weighted F1-Score: 0.43069500517611164
Mean Squared Error: 0.9439359267734554
Sensitivity (Recall) for each class: 0.351435495113656
Specificity for each class: [np.float64(0.6754966887417219), np.float64(0.6737804878048781), np.float64(0.7132352941176471)]
